# This is the end: Podsumowanie
---

Witaj na naszych ostatnich wspólnych laboratoriach! Dzisiaj czeka nas wielki finał, w którym połączymy wszystkie nabyte do tej pory umiejętności.

Pobierzemy z internetu surowe zbiory danych makroekonomicznych, oczyścimy je, połączymy, zagregujemy i wyliczymy korelacje.

Kiedy dane będą gotowe, stworzymy **trzy w pełni animowane wizualizacje**, z których każda będzie rygorystycznie korzystać innej filozofii projektowania graficznego:
1. **Styl Edwarda Tufte (Matplotlib)** – Maksymalizacja wskaźnika "Data-Ink ratio". Wykresy odarte z jakiegokolwiek szumu, czysty minimalizm, gdzie atrament służy wyłącznie pokazywaniu danych.
2. **Styl Cole Nussbaumer Knaflic (Seaborn + Storytelling)** – Technika "Push to the background" oraz "Direct Labeling". Skupimy uwagę widza na jednej konkretnej historii, ukrywając resztę w szarym tle.
3. **Styl Williama Rankina (Geopandas)** – Animowany kartogram w odpowiednim rzucie z zablokowaną skalą barw, pokazujący ewolucję świata na przestrzeni dekad.

Zaczynamy!

## Zadanie 0: Inżynieria Danych – Zbieranie i Agregacja
---

Zanim przejdziemy do pięknych, animowanych wykresów, musimy przygotować nasz materiał dowodowy. Proces ten analitycy nazywają **Data Preparation** (Przygotowaniem Danych).

**O co chodzi w tym zadaniu?**
Chcemy przeanalizować, jak bogactwo narodu (PKB) wpływa na to, jak długo żyją jego obywatele (Długość życia). Zbadamy to nie na podstawie jednego roku, ale w dynamicznym, historycznym ujęciu na przełomie dziesięcioleci. Najpierw pobierzemy światowe wskaźniki makroekonomiczne z cenionego projektu badawczego *Gapminder*, a potem znajdziemy w internecie mapę, do której te dane "przykleimy". Na koniec, jak na dobrych badaczy przystało, wyliczymy, czy te dwie rzeczy w ogóle są ze sobą powiązane (korelacja).

**Linki do wykorzystania:**
* Wbudowany w środowisko zbiór danych: `import plotly.express as px` a następnie wywołaj funkcję `px.data.gapminder()`.
* Mapa Świata w formacie GeoJSON: `"https://raw.githubusercontent.com/johan/world.geo.json/master/countries.geo.json"`

**Twoim zadaniem jest napisanie kodu, który:**
1. Wczyta tabelę statystyczną używając funkcji `px.data.gapminder()` i przypisze ją do zmiennej `df_data`.
2. Zmieni nazwę kolumny `lifeExp` na `life_exp`, a `gdpPercap` na `gdp` dla wygody (użyj `.rename()`).
3. Wyliczy korelację (Pearsona) pomiędzy PKB (`gdp`) a Długością Życia (`life_exp`) na podstawie całej historii i wyświetli ją za pomocą printa.
4. Sprawdzi, jak wyglądają średnie dla każdego kontynentu. Użyj `.groupby('continent')[['life_exp', 'gdp']].mean().round(1)` i wyświetl wynik.
5. Pobierze przestrzenną mapę świata wywołując `gpd.read_file()` na podanym wyżej linku do GeoJSON i zapisze do zmiennej `gdf_world`.
6. Odrzuci te rekordy z kolumny `id`, które wskazują Antarktydę (`'ATA'`) z `gdf_world` i wyświetli kształt połączonej bazy.

In [8]:
import plotly.express as px
import geopandas as gpd

df_data = px.data.gapminder()
df_data = df_data.rename(columns={'lifeExp': 'life_exp', 'gdpPercap': 'gdp'})

correlation = df_data['gdp'].corr(df_data['life_exp'])
print(f"Korelacja (Pearsona) pomiędzy PKB a Długością Życia: {correlation:.2f}")

continent_means = df_data.groupby('continent')[['life_exp', 'gdp']].mean().round(1)
print("\nŚrednie dla każdego kontynentu:")
print(continent_means)

geojson_url = "https://raw.githubusercontent.com/johan/world.geo.json/master/countries.geo.json"
gdf_world = gpd.read_file(geojson_url)

gdf_world = gdf_world[gdf_world['id'] != 'ATA']
print(df_data.shape)

Korelacja (Pearsona) pomiędzy PKB a Długością Życia: 0.58

Średnie dla każdego kontynentu:
           life_exp      gdp
continent                   
Africa         48.9   2193.8
Americas       64.7   7136.1
Asia           60.1   7902.2
Europe         71.9  14469.5
Oceania        74.3  18621.6
(1704, 8)


---
## Zadanie 1: Styl Edwarda Tufte – Minimalistyczna Animacja Matplotlib
---

**Wprowadzenie teoretyczne:**
Edward Tufte to amerykański statystyk i profesor emerytowany Uniwersytetu Yale, powszechnie uznawany za pioniera nowoczesnej wizualizacji danych. Sformułował on pojęcie **Data-Ink Ratio** (wskaźnik atramentu danych). Według Tuftego idealny wykres to taki, w którym niemal cały "atrament" (piksele na ekranie) służy bezpośredniej prezentacji danych. Wszelkie ozdobniki – siatki w tle, zbędne legendy, ozdobne obramowania – to tzw. "Chartjunk" (szum wizualny), który obciąża poznawczo odbiorcę, nie wnosząc żadnej wartości informacyjnej.

W tym zadaniu stworzymy animowany wykres punktowy (Scatter Plot) przedstawiający relację między zamożnością (PKB per capita na osi X) a zdrowiem (oczekiwaną długością życia na osi Y) na przestrzeni dziesięcioleci. Odrzucimy wszelkie zbędne elementy graficzne, dążąc do maksymalnego minimalizmu.

**Opis komend i parametrów, które wykorzystasz:**
* `ax.spines['left'].set_position(('outward', 10))` – ta unikalna metoda Matplotlib przesuwa fizyczną linię osi na zewnątrz o określoną liczbę punktów (tu o 10). Zapobiega to nakładaniu się skrajnych punktów danych na linie osi, co jest flagowym elementem stylu Tuftego.
* `ax.set_xscale('log')` – PKB per capita w krajach rozwiniętych rośnie wykładniczo i osiąga ogromne wartości, podczas gdy kraje rozwijające się mają bardzo niskie PKB. Skala logarytmiczna "rozszerza" lewą stronę wykresu, dzięki czemu uboższe kraje nie zostaną ściśnięte w jedną, nieczytelną plamę.
* `scat.set_offsets(dane)` – to niezwykle wydajna metoda animacji. Zamiast niszczyć i rysować punkty na nowo w każdej klatce, pobiera ona istniejący obiekt `PathCollection` (nasze kropki) i w ułamku milisekundy zmienia ich współrzędne na płótnie.
* `rok_text = ax.text(..., transform=ax.transAxes)` – parametr `transform` powoduje, że współrzędne tekstu (np. `0.05, 0.95`) są liczone jako procent szerokości i wysokości całego okna wykresu (od 0 do 1), a nie w wartościach mierzonych w PKB czy latach życia. Dzięki temu napis roku zawsze będzie stał w tym samym, bezpiecznym rogu.

---
**Instrukcja krok po kroku:**
* **Krok 1:** Wyciągnij ze zbioru `df_data` posortowaną listę unikalnych lat i przypisz ją do zmiennej `lata_animacji`. Chronologiczne posortowanie lat jest niezbędne do prawidłowego przebiegu animacji w czasie.
* **Krok 2:** Zainicjuj płótno za pomocą `plt.subplots(figsize=(10, 6))`. Wdroż zasady Tuftego: ukryj krawędzie osi (top i right), odsuń oś dolną i lewą o 10 punktów na zewnątrz, zmień skalę osi X na logarytmiczną oraz zablokuj zakresy za pomocą `set_xlim(200, 150000)` i `set_ylim(20, 90)`.
* **Krok 3:** Wygeneruj stan początkowy wykresu: stwórz pusty wykres punktowy `scat` o określonym rozmiarze (parametr `s=40`) i kolorze, bez obramowania punktów (`edgecolor='none'`). Przygotuj dynamiczną zmienną tekstową `rok_text` w lewym górnym rogu.
* **Krok 4:** Zdefiniuj funkcję aktualizującą `update_tufte(frame_rok)`. Funkcja ta dla każdej kolejnej klatki (roku) odfiltruje odpowiednie dane, zaktualizuje pozycje punktów metodą `.set_offsets()` i podmieni wyświetlany rok. Musi ona bezwzględnie zwrócić zmodyfikowane obiekty: `return scat, rok_text`.
* **Krok 5:** Zamknij statyczny, roboczy wykres poleceniem `plt.close()`, aby zapobiec powielaniu się statycznego rysunku pod gotową animacją w Google Colab.
* **Krok 6:** Skonfiguruj i uruchom `FuncAnimation` przekazując przygotowane płótno, funkcję aktualizującą, listę klatek i czas interwału (np. `interval=250` milisekund). Wyrenderuj animację jako interaktywny widget HTML.

***

**ZADANIE DLA CIEBIE (Dostosowanie estetyki):**
*Skonfiguruj ten wykres zgodnie ze swoją własną estetyką, dbając o czysty, minimalistyczny styl. Możesz zmodyfikować rozmiar kulek (`s`), dobrać własny odcień szarości lub grafitu dla tła i punktów (np. używając kodów HEX), zmienić czcionkę etykiet osi, a także zmodyfikować odległość odsunięcia osi (`outward`). Pamiętaj jednak o rygorystycznym zakazie stosowania jaskrawych kolorów i siatek w tle – wykres musi pozostać wzorem minimalizmu.*

In [12]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

lata_animacji = sorted(df_data['year'].unique())

fig, ax = plt.subplots(figsize=(10, 6))
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)
ax.spines['left'].set_position(('outward', 10))
ax.spines['bottom'].set_position(('outward', 10))
ax.set_xscale('log')
ax.set_xlim(200, 150000)
ax.set_ylim(20, 90)
ax.set_xlabel('PKB per capita (skala logarytmiczna)')
ax.set_ylabel('Długość życia')
ax.set_title('Relacja między PKB a Długością Życia na przestrzeni lat (Styl Tufte)', fontsize=14)

scat = ax.scatter([], [], s=40, c='#696969', edgecolor='none', alpha=0.7)
rok_text = ax.text(0.05, 0.95, '', transform=ax.transAxes, fontsize=18, verticalalignment='top', color='gray')

def update_tufte(frame_rok):
    df_current_year = df_data[df_data['year'] == frame_rok].copy()
    df_current_year.dropna(subset=['gdp', 'life_exp'], inplace=True)

    scat.set_offsets(df_current_year[['gdp', 'life_exp']].values)
    rok_text.set_text(str(frame_rok))
    return scat, rok_text

plt.close(fig)

anim = FuncAnimation(fig, update_tufte, frames=lata_animacji, interval=250, blit=True)
HTML(anim.to_jshtml())

---
## Zadanie 2: Styl Cole Nussbaumer Knaflic – Storytelling z Seaborn
---

**Wprowadzenie teoretyczne:**
Cole Nussbaumer Knaflic w swojej książce *Storytelling with Data* udowadnia, że surowe dane rzadko wywołują emocje i popychają do działania. Dobry analityk nie powinien zmuszać odbiorcy do domyślania się, co jest ważne na wykresie. Kluczem jest wyciszenie tła (szary kolor) i użycie jednego, silnego, przyciągającego wzrok kontrastowego koloru na elemencie, o którym chcemy opowiedzieć naszą historię (atrybut przeduwagowy).

Zbudujemy animowany wykres liniowy pokazujący, jak na przestrzeni lat rosła oczekiwana długość życia na różnych kontynentach. Zastosujemy technikę bezpośredniego etykietowania (Direct Labeling) – usuniemy domyślną legendę i umieścimy podpis kontynentu bezpośrednio na końcu ruchomej linii, co drastycznie obniża wysiłek umysłowy widza. Wybierzemy jeden kontynent (**Azję**), aby opowiedzieć o jego niebywałym sukcesie cywilizacyjnym.

**Opis komend i parametrów, które wykorzystasz:**
* `ax.clear()` – ponieważ biblioteka Seaborn rysuje złożone, wielowarstwowe obiekty graficzne, nie możemy łatwo przesunąć linii. Wewnątrz funkcji `update` musimy doszczętnie wyczyścić całą oś, narysować wykres na nowo i od nowa zdefiniować limity osi oraz napisy.
* `sns.lineplot(data=..., hue='continent', palette=['lightgray']*4, linewidth=1.5)` – ta komenda w Seaborn rysuje linie tła. Przekazując jako paletę listę czterech identycznych jasnoszarych kolorów, spychamy pozostałe kontynenty do mało widocznego tła.
* `ax.text(ostatni_x + 1, ostatni_y, "Azja")` – bezpośrednie etykietowanie. Tekst pojawi się dokładnie 1 rok na osi X za ostatnim zarejestrowanym punktem niebieskiej linii, na tej samej wysokości pionowej (Y).
* `ax.set_title("...", fontsize=16, fontweight='bold', loc='left', color='#333333')` – tytuł akcji (Action Title). Musi wprost opowiadać o tym, co widzimy, i być wyrównany do lewej krawędzi (loc='left'), co jest naturalnym punktem startowym czytania dla ludzkiego oka.

---
**Instrukcja krok po kroku:**
* **Krok 1:** Przygotuj dane – stwórz nową tabelę `df_kontynenty`, wyliczając średnią długość życia dla każdego kontynentu w poszczególnych latach (użyj `groupby` oraz `mean()`).
* **Krok 2:** Zainicjuj płótno i osie Matplotlib o rozmiarze `(12, 7)`.
* **Krok 3:** Zdefiniuj funkcję aktualizującą `update_swd(frame_rok)`. Na samym początku wyczyść oś za pomocą `ax.clear()` i odfiltruj z tabeli `df_kontynenty` dane historyczne od początku do aktualnie renderowanego roku `frame_rok`.
* **Krok 4:** Narysuj tło w Seaborn: wyrenderuj linie dla wszystkich kontynentów oprócz Azji, nadając im cienki, jasnoszary kolor. Następnie narysuj linię dla Azji na grubo, używając mocnego koloru (np. `'steelblue'`). Na samym końcu linii Azji dodaj dynamiczny podpis tekstowy podążający za wykresem.
* **Krok 5:** Przywróć wyczyszczone ustawienia osi: stałe limity osi X i Y (odpowiednio `1950-2025` i `35-85`), usuń górną i prawą ramkę oraz zdefiniuj lewostronny, pogrubiony **Action Title** opowiadający historię sukcesu Azji.
* **Krok 6:** Zamknij wykres statyczny i wygeneruj animację `FuncAnimation` dla unikalnych lat, a następnie wyświetl ją jako odtwarzacz wideo.

***

**ZADANIE DLA CIEBIE (Dostosowanie estetyki):**
*Skonfiguruj ten wykres zgodnie ze swoją własną estetyką i wyczuciem kompozycji. Możesz wybrać inny kontynent jako głównego bohatera swojej opowieści (np. Afrykę lub Amerykę), dobrać dla niego inny, wyrazisty kolor (np. krwistą czerwień `'crimson'`, głęboką zieleń `'forestgreen'` lub ciepły pomarańcz), zmodyfikować odcienie szarości dla linii tła, zmienić ich grubość (`linewidth`), a także sformułować własny, unikalny i angażujący tytuł akcji (Action Title) oddający charakter wybranego kontynentu.*

In [17]:
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import pandas as pd

df_kontynenty = df_data.groupby(['year', 'continent'])['life_exp'].mean().reset_index()

fig, ax = plt.subplots(figsize=(12, 7))

def update_swd(frame_rok):
    ax.clear()
    df_history = df_kontynenty[df_kontynenty['year'] <= frame_rok]

    sns.lineplot(
        data=df_history[df_history['continent'] != 'Africa'],
        x='year',
        y='life_exp',
        hue='continent',
        palette=['lightgray']*4,
        linewidth=1.5,
        ax=ax,
        legend=False
    )

    sns.lineplot(
        data=df_history[df_history['continent'] == 'Africa'],
        x='year',
        y='life_exp',
        color='steelblue',
        linewidth=3,
        ax=ax,
        legend=False
    )

    africa_last_point = df_history[(df_history['continent'] == 'Africa') & (df_history['year'] == frame_rok)]
    if not africa_last_point.empty:
        ostatni_x = africa_last_point['year'].values[0]
        ostatni_y = africa_last_point['life_exp'].values[0]
        ax.text(ostatni_x + 0.5, ostatni_y, "Azja",
                color='steelblue', fontsize=12, fontweight='bold', ha='left', va='center')

    ax.set_xlim(1950, 2010)
    ax.set_ylim(35, 85)
    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)
    ax.set_xlabel('Rok')
    ax.set_ylabel('Oczekiwana długość życia')
    ax.set_title(
        f"Wzrost długości życia w Afryce do roku {frame_rok}.",
        fontsize=16,
        fontweight='bold',
        loc='left',
        color='#333333'
    )

    return fig,

plt.close(fig)

anim_swd = FuncAnimation(
    fig,
    update_swd,
    frames=sorted(df_kontynenty['year'].unique()),
    interval=250,
    blit=True
)

HTML(anim_swd.to_jshtml())

---
## Zadanie 3: Radykalna Kartografia – Animowany Kartogram Geopandas
---

**Wprowadzenie teoretyczne:**
Tradycyjna mapa statyczna to zaledwie jedna stop-klatka w historii świata. Prawdziwą potęgę wizualizacji geograficznej odkrywamy, gdy ożywimy mapę za pomocą czasu. Stworzymy animowany kartogram, pokazujący jak na przestrzeni dekad (od roku 1952 do 2007) zmieniała się oczekiwana długość życia mieszkańców naszej planety.

W tworzeniu animowanych map kluczową rolę odgrywa naukowa rzetelność wizualizacji. Musimy bezwzględnie zablokować skalę barwną kartogramu za pomocą parametrów `vmin` i `vmax`. Jeśli tego nie zrobimy, biblioteka w każdej klatce dopasuje kolory na nowo (np. najjaśniejszy odcień żółci w 1952 roku oznaczałby 55 lat życia, a w 2007 roku oznaczałby 80 lat). Taki błąd – tzw. kłamstwo kartograficzne – uniemożliwiłby jakąkolwiek poprawną interpretację zmian w czasie.

**Opis komend i parametrów, które wykorzystasz:**
* `gdf_world_clean.to_crs(epsg=3395)` – transformacja układu współrzędnych do rzutu walcowego Mercatora. Zapobiega to nienaturalnemu "spłaszczeniu" kontynentów, z którym mamy do czynienia w surowym formacie WGS84.
* `vmin` i `vmax` – parametry w funkcji `.plot()`. Definiują sztywny punkt startowy i końcowy naszej legendy (odpowiednio `30` i `85` lat), zamrażając znaczenie kolorów na całym dystansie animacji.
* `missing_kwds={'color': 'lightgrey'}` – genialny argument dla funkcji `.plot()`. Pozwala automatycznie zepchnąć do estetycznego, jasnoszarego tła kraje, dla których w danym roku nie odnotowano żadnych pomiarów statystycznych, zamiast zostawiać na ich miejscu dziury w mapie.
* `ax.set_axis_off()` – całkowicie ukrywa osie, ramki i współrzędne geograficzne na brzegach mapy, dzięki czemu prezentuje się ona czysto i nowocześnie.

---
**Instrukcja krok po kroku:**
* **Krok 1:** Przygotuj mapę świata – odrzuć Antarktydę (`'ATA'`), aby mapa była bardziej zwarta w pionie, i przetransformuj układ współrzędnych do rzutu Mercatora (`EPSG:3395`).
* **Krok 2:** Zdefiniuj sztywne, globalne punkty skrajne dla naszej palety barw: `vmin_life = 30` oraz `vmax_life = 85` (reprezentujące minimalne i maksymalne lata życia w badanej historii).
* **Krok 3:** Zainicjuj duże płótno `fig, ax` o rozmiarach `(12, 7)`.
* **Krok 4:** Zdefiniuj funkcję aktualizującą `update_map(frame_rok)`. Oczyść oś na starcie (`ax.clear()`), odfiltruj dane z `df_data` dla konkretnego roku i połącz (Merge) naszą mapę świata z danymi statystycznymi na pasujących kolumnach (`'id'` oraz `'iso_alpha'`).
* **Krok 5:** Wyrysuj mapę za pomocą metody `.plot()`. Przekaż do niej: zablokowane zakresy skali kolorystycznej, paletę `'magma'`, szary kolor dla brakujących danych oraz czarny, bardzo cienki obrys granic państw (`linewidth=0.2`). Wyłącz osie i narysuj duży licznik roku na środku na samym dole mapy.
* **Krok 6:** Zamknij wykres roboczy i wygeneruj animację przestrzenną.

***

**ZADANIE DLA CIEBIE (Dostosowanie estetyki):**
*Skonfiguruj tę mapę zgodnie ze swoimi własnymi preferencjami estetycznymi. Możesz dobrać zupełnie inną paletę kolorów (np. klasyczną `'viridis'`, ognistą `'inferno'` lub oceaniczną `'YlGnBu'`), zmienić kolor granic państw (`edgecolor`) oraz ich grubość (`linewidth`), wybrać inny kolor dla państw bez danych (np. ciemniejszy szary lub pastelowy beż), a także zmienić pozycję, krój oraz rozmiar czcionki wyświetlanego licznika lat.*

In [22]:
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import geopandas as gpd

gdf_world_clean = gdf_world[gdf_world['id'] != 'ATA']
gdf_world_clean = gdf_world_clean.to_crs(epsg=3395)

vmin_life = 30
vmax_life = 85

fig, ax = plt.subplots(1, 1, figsize=(12, 7))

def update_map(frame_rok):
    ax.clear()
    df_year = df_data[df_data['year'] == frame_rok]
    gdf_merged = gdf_world_clean.merge(df_year, left_on='id', right_on='iso_alpha', how='left')

    gdf_merged.plot(
        column='life_exp',
        cmap='viridis',
        linewidth=0.2,
        ax=ax,
        edgecolor='black',
        vmin=vmin_life,
        vmax=vmax_life,
        legend=False,
        missing_kwds={'color': 'lightgrey'}
    )

    ax.set_axis_off()
    ax.set_title(str(frame_rok), fontsize=25, color='#333333', loc='center', y=0.05)

    return fig,

plt.close(fig)

anim_geo = FuncAnimation(
    fig,
    update_map,
    frames=sorted(df_data['year'].unique()),
    interval=500,
    blit=True
)

HTML(anim_geo.to_jshtml())

In [ ]:
zmienna = [15,16,17,18,20,21,66,67,68,70,71,72,73,91,92,95,109,110,111]
nowe = []
for z in zmienna:
  nowe.append(z-14)
